# K 近邻（KNN）

非参数方法：不显式训练权重，预测时看 `k` 个最近邻的标签投票。  
全量 `60000` 训练样本在 CPU 上较慢，可在下方变量里**先取子集**做练习。


In [1]:
from pathlib import Path

import numpy as np
import torchvision
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier


## 1. 超参数与是否只用部分训练数据


In [2]:
# 设为例如 10000 可明显加速；None 表示全量 60000
TRAIN_SUBSET = None  # e.g. 10000
TEST_N = 2000


## 2. 读入数据并拉平为 784 维向量

优先 `data/raw`；否则从 torchvision 下载目录读取。


In [3]:
from pathlib import Path

import torchvision
from mnist_from_raw import load_all_numpy, raw_files_available

if raw_files_available():
    tr_x, tr_y, te_x, te_y = load_all_numpy()
    train_x = tr_x.astype(np.float32) / 255.0
    train_y = tr_y
    test_x_img = te_x[:TEST_N].astype(np.float32) / 255.0
    test_y = te_y[:TEST_N]
    print("数据来源: data/raw")
else:
    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())
    train_ds = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT), train=True, download=download
    )
    test_ds = torchvision.datasets.MNIST(
        root=str(MNIST_ROOT), train=False, download=download
    )
    train_x = train_ds.data.numpy().astype(np.float32) / 255.0
    train_y = train_ds.targets.numpy()
    test_x_img = test_ds.data.numpy()[:TEST_N].astype(np.float32) / 255.0
    test_y = test_ds.targets.numpy()[:TEST_N]
    print("数据来源: torchvision ->", MNIST_ROOT.resolve())

if TRAIN_SUBSET is not None:
    train_x, train_y = train_x[:TRAIN_SUBSET], train_y[:TRAIN_SUBSET]

train_x = train_x.reshape(len(train_x), -1)
test_x = test_x_img.reshape(len(test_x_img), -1)
print("train:", train_x.shape, "test:", test_x.shape)


数据来源: data/raw
train: (60000, 784) test: (2000, 784)


## 3. 在验证集上选 k（这里用测试集代替验证，仅演示流程）

对奇数 `k` 扫一遍，选准确率最高的 `k`。


In [4]:
k_vals = list(range(1, 30, 2))
accuracies = []
for k in k_vals:
    clf = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    clf.fit(train_x, train_y)
    acc = clf.score(test_x, test_y)
    accuracies.append(acc)
    print(f"k={k:2d}  acc={acc:.4f}")

best_i = int(np.argmax(accuracies))
best_k = k_vals[best_i]
print(f"best k={best_k}  acc={accuracies[best_i]:.4f}")


k= 1  acc=0.9600
k= 3  acc=0.9580
k= 5  acc=0.9565
k= 7  acc=0.9565
k= 9  acc=0.9500
k=11  acc=0.9505
k=13  acc=0.9485
k=15  acc=0.9450
k=17  acc=0.9480
k=19  acc=0.9485
k=21  acc=0.9470
k=23  acc=0.9460
k=25  acc=0.9435
k=27  acc=0.9430
k=29  acc=0.9405
best k=1  acc=0.9600


## 4. 用最佳 k 在全训练集上拟合并输出分类报告


In [5]:
model = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
model.fit(train_x, train_y)
pred = model.predict(test_x)
print(classification_report(test_y, pred, digits=4))


              precision    recall  f1-score   support

           0     0.9721    0.9943    0.9831       175
           1     0.9707    0.9915    0.9810       234
           2     0.9860    0.9635    0.9746       219
           3     0.9431    0.9614    0.9522       207
           4     0.9583    0.9539    0.9561       217
           5     0.9297    0.9609    0.9451       179
           6     0.9775    0.9775    0.9775       178
           7     0.9415    0.9415    0.9415       205
           8     0.9778    0.9167    0.9462       192
           9     0.9430    0.9381    0.9406       194

    accuracy                         0.9600      2000
   macro avg     0.9600    0.9599    0.9598      2000
weighted avg     0.9602    0.9600    0.9600      2000

